# Visualizing Graph Representation

{py:mod}`claveles.graph.visualize` draws a graph representation on the macronode lattice, which is how you
check what a conversion produced and which macronode each mode passes through.

* {py:func}`~claveles.graph.visualize.make_figure` returns a {py:class}`matplotlib.figure.Figure`, which
  Jupyter displays and {py:func}`matplotlib.pyplot.show` shows.
* {py:func}`~claveles.graph.visualize.savefig` writes one straight to a file.

Both take the same keyword options, listed in {ref}`sec:viz-graph-options`.

In [ ]:
# Draw the figures on this page as SVG so that they stay sharp at any zoom level.
# The cell itself is hidden from the built page by the `remove-cell` tag.
%config InlineBackend.figure_formats = ["svg"]

## A sample graph

In [ ]:
import copy

from claveles.graph import GraphRepr
from claveles.graph.constant import BLANK_MODE

graph = GraphRepr(5, 5)

In [ ]:
from math import pi

from claveles.feedforward import feedforward
from claveles.graph import Wiring, ops


@feedforward
def add_pi(x: float) -> float:
    from math import pi

    return x + pi


@feedforward
def square(x: float) -> float:
    return x * x


@feedforward
def cube(x: float) -> float:
    return x * x * x


graph.place_operation(ops.Initialization((0, 1), 0.0, (0, BLANK_MODE)))
graph.place_operation(ops.Initialization((1, 0), 0.0, (BLANK_MODE, 1)))
graph.place_operation(ops.Initialization((2, 0), 0.0, (BLANK_MODE, 2)))
graph.place_operation(ops.BeamSplitter((1, 1), 0.0, 0.0, displacement_k_minus_n=(1.0, -1.0), swap=True))
graph.place_operation(ops.TwoModeShear((2, 1), 0.0, 0.0, displacement_k_minus_1=(-1.0, 1.0), swap=True))
graph.place_operation(ops.PhaseRotation((3, 1), pi / 2, swap=True))
graph.place_operation(ops.Measurement((1, 2), 0.0))  # Measure mode 0
m0 = graph.get_mode_measured_value(0)
graph.place_operation(Wiring((2, 2), swap=True))
graph.place_operation(ops.ControlledZ((3, 2), 0.0, swap=False))
graph.place_operation(ops.Measurement((4, 2), 0.0))  # Measure mode 1
m1 = graph.get_mode_measured_value(1)
graph.place_operation(
    ops.ShearPInvariant((3, 3), square(m0), displacement_k_minus_n=(add_pi(m0), cube(m1)), swap=True)
)
graph.place_operation(ops.Measurement((4, 3), 0.0))  # Measure mode 2

Drawing it:

In [ ]:
from claveles.graph.visualize import make_figure

make_figure(graph);

{py:func}`~claveles.graph.visualize.savefig` writes the same figure to a file:

In [ ]:
from claveles.graph.visualize import savefig

savefig(graph, "claveles_example.png")

(sec:viz-graph-options)=

## Options

### The lattice and the figure


| Option | Default | Description |
| --- | --- | --- |
| `title` | `""` | Title written above the figure. Empty means no title. |
| `scale` | `2.0` | Scale of the figure size. Raise it when the image comes out too small to read. |
| `fontsize` | `10.0` | Font size of every label except the feedforward ones. |
| `macronode_radius` | `1.0` | Radius of the large circle standing for a macronode. |
| `macronode_edge_color` | `lightgray` | Colour of that circle's outline. |
| `micronode_radius` | `0.2` | Radius of the four small circles inside a macronode. |
| `micronode_edge_color` | `lightgray` | Colour of their outlines. |
| `highlight_macronode_edge_color` | `{}` | Macronodes to outline in a colour of their own, as a dictionary from `(h, w)` coordinate to colour. Empty means none. |
| `show_displacement` | `True` | Whether the displacement applied to each teleported mode is written beside it. |
| `displacement_text_format` | a two-row column vector | Format string the two displacements are written with, taking the `x` and `p` components in that order. A component that comes from a measurement is written as the symbol standing for it (`d_x^{-1}`, `d_p^{-N}`, and so on) rather than as a number. |
| `show_feedforward` | `True` | Whether feedforward is drawn at all. |
| `show_feedforward_param_label` | `True` | Whether each feedforward arrow is labelled with the parameter it drives. |
| `feedforward_param_fontsize` | `8.0` | Font size of that label. |
| `feedforward_arrow_style` | `->` | Matplotlib arrow style of the feedforward arrows. |
| `feedforward_line_style` | `-` | Line style of the feedforward connections. |
| `feedforward_arrow_color` | `lightgreen` | Their colour, in any format matplotlib accepts. |

### The operations

| Option | Default | Description |
| --- | --- | --- |
| `show_op_params` | `False` | Whether each operation's parameters are written beside its macronode. |
| `operation_colors` | one colour per operation | Fill colour of a macronode, as a dictionary from operation type to colour. By default measurement and initialization are `black`, wiring is `white`, and the gates take the rest. A type left out of the dictionary is logged as a warning and drawn in its default colour. |
| `operation_description` | `$M$`, `$R$`, … | Symbol written beside a macronode, as a dictionary from operation type to text. Wiring is labelled with an empty string, which is what keeps a graph full of wiring readable. As with `operation_colors`, a type left out of it is logged as a warning and labelled with its default. |
| `operation_params_format` | `$({:.2f})$` and friends | Format string the parameters are written with when `show_op_params=True`. It needs one field per parameter the operation takes. |
| `operation_params_symbol_format` | `$\phi$`, `$\theta$`, … | Symbol standing for one parameter, as a dictionary from `(operation type, parameter index)`. A parameter whose value comes from a measurement has no number to print, so this symbol is written in its place and labels the feedforward arrow as well. |

### The modes

| Option | Default | Description |
| --- | --- | --- |
| `mode_colors` | a colour per mode | Colours of the modes, as a list indexed by mode. |
| `mode_none_color` | `lightgray` | Colour of a micronode carrying no mode. |
| `mode_none_linestyle` | `:` | Line style of those. |
| `show_mode_index` | `False` | Whether each mode's index is written on it. |
| `ignore_wiring` | `False` | Whether the figure is cropped to the block from the top left that holds every macronode other than straight-through wiring: the rows below it and the columns to its right are dropped, and an all-wiring row or column inside it stays. A graph laid out for the machine is mostly wiring, so this is what makes a compiled graph legible. |

The examples below vary them one or two at a time.

`title` puts a caption above the figure:

In [ ]:
make_figure(graph, title="example");

`scale` grows the whole figure; raise it when the image comes out too small to read:

In [ ]:
make_figure(graph, scale=3.0);

`fontsize` sets every label except the feedforward ones:

In [ ]:
make_figure(graph, fontsize=15.0);

`macronode_radius`:

In [ ]:
make_figure(graph, macronode_radius=2.0);

`macronode_edge_color`:

In [ ]:
make_figure(graph, macronode_edge_color="brown");

`micronode_radius`:

In [ ]:
make_figure(graph, micronode_radius=0.28);

`micronode_edge_color`:

In [ ]:
make_figure(graph, micronode_edge_color="brown");

`highlight_macronode_edge_color` outlines the macronodes you name, and only those:

In [ ]:
make_figure(graph, highlight_macronode_edge_color={(0, 0): "red", (1, 1): "blue"});

`show_op_params` writes each operation's parameters beside it:

In [ ]:
make_figure(graph, show_op_params=True);

`mode_none_color` recolours the micronodes carrying no mode:

In [ ]:
make_figure(graph, mode_none_color="black");

`mode_none_linestyle` restyles their outline:

In [ ]:
make_figure(graph, mode_none_linestyle="-");

`mode_colors` is indexed by mode:

In [ ]:
make_figure(graph, mode_colors=["red", "green", "blue"]);

`show_displacement=False` drops the displacement labels:

In [ ]:
make_figure(graph, show_displacement=False);

`ignore_wiring=True` drops the rows at the bottom and the columns at the right that carry nothing but wiring. That matters most for a
graph compiled to the machine's size, which is mostly wiring; the cell below reproduces such a graph with
{py:class}`~claveles.client.OptQCCompiler`, as the client would before execution.

In [ ]:
from claveles.client import OptQCCompiler
from claveles.graph.visualize import make_figure

# The client compiles a graph to the target hardware size before execution.
# We reproduce that compiled graph locally with OptQCCompiler.
compiler = OptQCCompiler()
compiled_graph = copy.deepcopy(graph)
compiled_graph.increase_local_macronodes(compiler.n_local_macronodes)
make_figure(compiled_graph, ignore_wiring=True);

`show_mode_index` writes each mode's index on it:

In [ ]:
make_figure(graph, show_mode_index=True, ignore_wiring=True);

Options combine, of course:

In [ ]:
make_figure(
    graph,
    macronode_edge_color="black",
    micronode_radius=0.1,
    mode_colors=["red", "blue", "green"],
    ignore_wiring=True,
);

{py:func}`~claveles.graph.visualize.savefig` takes all of them too:

In [ ]:
savefig(
    graph,
    "claveles_example_large.png",
    macronode_edge_color="black",
    micronode_radius=0.1,
    mode_colors=["red", "blue", "green"],
    ignore_wiring=True,
)